# 05 - Reglas simples (líneas base)

Antes de entrenar un modelo tenemos que saber **contra qué compararlo**. Si un modelo complicado no le gana a una regla que se puede hacer a mano, no sirve de nada. En este notebook probamos reglas muy simples, que no aprenden nada o casi nada, y vemos cuál es el error a superar.

Ejemplo de cómo se lee la tabla: queremos predecir la línea L, el día D, a la hora H.

| Regla | Qué predice | ¿Usa train? |
|---|---|---|
| `media_train` | El promedio de pasajeros de todas las filas de train | Sí (es un solo número) |
| `perfil_historico` | El promedio de train para esa línea, esa hora y ese tipo de día | Sí (es una tabla de promedios) |
| `ayer` | Los pasajeros de la línea L a la hora H del día anterior | No |
| `semana_pasada` | Los pasajeros de la línea L a la hora H de 7 días antes | No |
| `prom_4sem` | El promedio de 7, 14, 21 y 28 días antes, a la hora H | No |
| `mediana_4sem` | La mediana de esos mismos 4 días | No |

**La regla a superar es `mediana_4sem`.** En el EDA fue la variable que más se parecía a `pasajeros` (correlación 0,972). La regresión del notebook 06 también se compara contra ella.

### Cómo medimos el error

En las fórmulas, $y$ es lo que pasó de verdad, $\hat y$ (se lee "y sombrero") es lo que predijimos y $N$ es la cantidad de filas.

| Métrica | Fórmula | Qué quiere decir |
|---|---|---|
| MAE | $\frac{1}{N}\sum \lvert \hat y - y \rvert$ | En promedio, por cuántos pasajeros le erramos en cada hora. Si da 37, le erramos por 37 pasajeros |
| RMSE | $\sqrt{\frac{1}{N}\sum (\hat y - y)^2}$ | Parecido al MAE, pero como eleva al cuadrado, los errores grandes pesan mucho más. Siempre da mayor o igual que el MAE |
| R² | $1 - \frac{\sum (\hat y - y)^2}{\sum (y - \bar y)^2}$ | Compara contra predecir siempre el promedio. 1 es perfecto, 0 es igual que el promedio, negativo es peor que el promedio |
| WAPE | $\frac{\sum \lvert \hat y - y \rvert}{\sum y}$ | El error total dividido por los pasajeros totales. 0,14 quiere decir que le erramos en el 14 % de los pasajeros |
| Sesgo | $\frac{1}{N}\sum (\hat y - y)$ | Hacia qué lado nos equivocamos. Positivo: predecimos de más. Negativo: predecimos de menos |
| Mejora | $1 - \frac{\text{MAE}}{\text{MAE de la mediana 4 sem}}$ | Cuánto menos error que la regla a superar. 0,10 es un 10 % menos de error. Negativo es peor |

**Intervalo de confianza del 95 %.** El MAE depende de qué días tocaron en validación. Para saber cuánto podría cambiar, sorteamos días al azar muchas veces (1000) y calculamos el MAE cada vez. Eso se llama **bootstrap**. El intervalo es el rango donde cae el 95 % de esos MAE. Sorteamos **días** y no filas porque las filas de un mismo día se parecen mucho (mismo clima, mismo feriado). No son datos independientes.

**Todo se mide en validación** (todo 2025), solo con filas completas. Test (2026) no se toca.

In [0]:
%run ./00_setup

In [0]:
import importlib
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

from bondis import metricas, variables

importlib.reload(metricas)
importlib.reload(variables)

spark.conf.set("spark.sql.session.timeZone", config.TIMEZONE)
plt.rcParams.update({"figure.figsize": (13, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 200)

REFERENCIA = "mediana_4sem"

gold = variables.con_tipo_dia(spark.table(config.TABLE_GOLD)).where(F.col("fila_completa"))
train = gold.where(F.col("particion") == "train")
val = gold.where(F.col("particion") == "val")

r = val.agg(F.count("*").alias("filas"), F.countDistinct("fecha").alias("dias"),
            F.min("fecha").alias("desde"), F.max("fecha").alias("hasta")).first()
print(f"Validación: {r['filas']:,} filas, {r['dias']} días ({r['desde']} a {r['hasta']})")

## 1. Calcular las reglas que usan train

- `media_train`: un solo número, el promedio de todo train.
- `perfil_historico`: una tabla con el promedio por línea, hora y tipo de día. Si una combinación no aparece en train (por ejemplo, una línea que nunca anduvo un feriado a esa hora), se usa el promedio de esa línea a esa hora.

In [0]:
media_train = train.agg(F.avg("pasajeros")).first()[0]
perfil = train.groupBy("linea", "hora", "tipo_dia").agg(F.avg("pasajeros").alias("_perfil"))
perfil_lh = train.groupBy("linea", "hora").agg(F.avg("pasajeros").alias("_perfil_lh"))
tamanos = variables.tamano_linea(train)

val_pred = (
    val.join(perfil, ["linea", "hora", "tipo_dia"], "left")
    .join(perfil_lh, ["linea", "hora"], "left")
    .join(tamanos, "linea", "left")
    .withColumn("pred_perfil", F.coalesce("_perfil", "_perfil_lh"))
    .withColumn("pred_media", F.lit(media_train))
    .fillna({"tamano_linea": "chica"})
)

PREDICCIONES = {
    "media_train": "pred_media",
    "perfil_historico": "pred_perfil",
    "ayer": "pasajeros_lag_1d",
    "semana_pasada": "pasajeros_lag_7d",
    "prom_4sem": "pasajeros_prom_4sem",
    "mediana_4sem": "pasajeros_mediana_4sem",
}

print(f"media_train = {media_train:.1f} pasajeros por hora")
sin_perfil = val_pred.where(F.col("_perfil").isNull()).count()
print(f"Filas de val sin combinación (línea, hora, tipo de día) en train: {sin_perfil:,}")

## 2. Resultados en validación

Primero se calculan unas sumas por día para las seis reglas, en una sola pasada por los datos. Todas las métricas salen de esas sumas (ver `src/bondis/metricas.py`).

In [0]:
est = metricas.estadisticos_por_dia(val_pred, "pasajeros", PREDICCIONES, grupos=["tipo_dia", "tamano_linea"])

globales = metricas.agregar_mejora(metricas.metricas(est), REFERENCIA)
intervalos = metricas.bootstrap_dias(est, REFERENCIA, n_rep=1000, seed=config.SEED)
globales = globales.merge(intervalos, on="modelo")
orden = list(PREDICCIONES)
globales = globales.set_index("modelo").loc[orden].reset_index()


def fmt_intervalo(v, inf, sup, dec=1, pct=False):
    f = (lambda x: f"{100 * x:.{dec}f} %") if pct else (lambda x: f"{x:,.{dec}f}")
    return f"{f(v)} [{f(inf)} a {f(sup)}]"


vista = pd.DataFrame({
    "modelo": globales["modelo"],
    "MAE [IC 95 %]": [fmt_intervalo(*x) for x in globales[["mae", "mae_inf", "mae_sup"]].to_numpy()],
    "RMSE [IC 95 %]": [fmt_intervalo(*x) for x in globales[["rmse", "rmse_inf", "rmse_sup"]].to_numpy()],
    "R²": globales["r2"].round(3),
    "WAPE": (100 * globales["wape"]).round(1).astype(str) + " %",
    "sesgo": globales["sesgo"].round(1),
    "mejora [IC 95 %]": [fmt_intervalo(*x, pct=True) for x in globales[["mejora_mae", "mejora_mae_inf", "mejora_mae_sup"]].to_numpy()],
})
print(vista.to_string(index=False))

**Cómo leer la tabla.**
- El R² de `media_train` tiene que dar cerca de 0 (o un poco negativo, porque es el promedio de train y no el de val). Si las otras reglas tienen R² cerca de 1, se confirma lo que vimos en el EDA: el R² no sirve para elegir entre ellas, porque todas dan alto. La columna que importa es la **mejora** sobre la mediana de 4 semanas.
- Si el intervalo de la mejora no incluye el 0, la diferencia es real. Si lo incluye, podría ser casualidad de qué días tocaron.

## 3. ¿Dónde se equivoca cada regla?

### 3.1 Por tipo de día

Lo que esperamos: las reglas que miran semanas anteriores se equivocan en los días que no se parecen a esas semanas (feriados). En cambio, `perfil_historico` se equivoca cuando cambia el nivel general (por ejemplo, cuando terminan las clases en diciembre).

In [0]:
por_tipo = metricas.agregar_mejora(metricas.metricas(est, por=["tipo_dia"]), REFERENCIA, por=["tipo_dia"])
dias_tipo = est.groupby("tipo_dia")["fecha"].nunique().rename("días")
tipos = [t for t in variables.TIPOS_DIA if t in dias_tipo.index]

print("MAE por tipo de día (pasajeros por hora):")
t = por_tipo.pivot(index="modelo", columns="tipo_dia", values="mae").loc[orden, tipos].round(1)
print(pd.concat([dias_tipo[tipos].to_frame().T, t]).to_string(), "\n")

print("Sesgo por tipo de día (> 0: se pasa):")
print(por_tipo.pivot(index="modelo", columns="tipo_dia", values="sesgo").loc[orden, tipos].round(1).to_string())

### 3.2 Por tamaño de línea

Separamos las líneas según cuántos pasajeros llevaron en train: las 20 más grandes, las 30 que siguen y el resto. En las líneas grandes los errores son más grandes en pasajeros (ver EDA, sección 4), así que el MAE total lo deciden ellas. El WAPE, como es un porcentaje, sirve para comparar grupos de distinto tamaño.

In [0]:
por_tam = metricas.metricas(est, por=["tamano_linea"])
grupos_tam = [g for g in ["grande", "mediana", "chica"] if g in set(por_tam["tamano_linea"])]
for col, titulo in (("mae", "MAE"), ("wape", "WAPE")):
    t = por_tam.pivot(index="modelo", columns="tamano_linea", values=col).loc[orden, grupos_tam]
    print(f"{titulo} por tamaño de línea:")
    print((t * 100).round(1).astype(str).add(" %").to_string() if col == "wape" else t.round(1).to_string(), "\n")

### 3.3 Día por día

El MAE de cada día de validación para tres reglas. Los picos son los días en que la regla se equivoca mucho. Tendrían que coincidir con días especiales (líneas grises punteadas) o con cambios en las clases (líneas rojas: el primer día en que empiezan o terminan las clases de escuela o liceo).

In [0]:
info_dias = (val.select("fecha", "tipo_dia", "nombre_dia_especial", "clases_primaria", "clases_media")
             .distinct().toPandas())
info_dias["fecha"] = pd.to_datetime(info_dias["fecha"])

diario = metricas.metricas(est, por=["fecha"])
fig, ax = plt.subplots()
for m in ["semana_pasada", "mediana_4sem", "perfil_historico"]:
    s = diario[diario["modelo"] == m].sort_values("fecha")
    ax.plot(s["fecha"], s["mae"], marker=".", ms=4, lw=1, label=m)
tope = ax.get_ylim()[1]
for _, d in info_dias.dropna(subset=["nombre_dia_especial"]).iterrows():
    ax.axvline(d["fecha"], color="gray", ls=":", lw=1)
    ax.text(d["fecha"], tope, d["nombre_dia_especial"], rotation=90, va="top", ha="right", fontsize=7)

# Días en que empiezan o terminan las clases (clases_primaria o clases_media cambia respecto del día anterior)
cal = info_dias.sort_values("fecha").set_index("fecha")[["clases_primaria", "clases_media"]]
for nivel in ("primaria", "media"):
    c = cal[f"clases_{nivel}"].astype(int).diff()
    for f, v in c[c != 0].dropna().items():
        etiqueta = f"{'inicio' if v > 0 else 'fin'} de clases ({nivel})"
        ax.axvline(f, color="tab:red", ls="--", lw=1)
        ax.text(f, tope, etiqueta, rotation=90, va="top", ha="right", fontsize=7, color="tab:red")

ax.set(title="MAE por día en validación", ylabel="MAE (pasajeros por hora)")
ax.legend(loc="center left")
plt.show()

peores = (diario[diario["modelo"] == REFERENCIA].nlargest(10, "mae")
          .merge(info_dias, on="fecha")[["fecha", "tipo_dia", "nombre_dia_especial", "clases_primaria",
                                         "clases_media", "mae", "sesgo"]])
print(f"Los 10 días con más error de {REFERENCIA}:")
print(peores.round(1).to_string(index=False))

## 4. Guardar los resultados

Los resultados se guardan en la tabla `resultados_modelos` con `etapa = 'baseline'`. El notebook 06 agrega los suyos con otra etapa, así comparamos todo desde una sola tabla. Si se vuelve a correr este notebook, se reemplazan solo las filas de `baseline`.

In [0]:
ETAPA = "baseline"
salida = globales.assign(etapa=ETAPA, conjunto="val", referencia=REFERENCIA,
                         fecha_ejecucion=datetime.now().strftime("%Y-%m-%d %H:%M"))
columnas = ["etapa", "conjunto", "modelo", "referencia", "filas", "mae", "mae_inf", "mae_sup", "rmse",
            "rmse_inf", "rmse_sup", "r2", "wape", "sesgo", "mejora_mae", "mejora_mae_inf", "mejora_mae_sup",
            "fecha_ejecucion"]
salida = salida[columnas].astype({c: "float64" for c in columnas if c not in
                                  ("etapa", "conjunto", "modelo", "referencia", "fecha_ejecucion", "filas")})

escritor = spark.createDataFrame(salida).write.format("delta").mode("overwrite").option("mergeSchema", "true")
if spark.catalog.tableExists(config.TABLE_RESULTADOS):
    escritor.option("replaceWhere", f"etapa = '{ETAPA}'").saveAsTable(config.TABLE_RESULTADOS)
else:
    escritor.saveAsTable(config.TABLE_RESULTADOS)

print(spark.table(config.TABLE_RESULTADOS).where(F.col("etapa") == ETAPA)
      .select("modelo", "mae", "rmse", "r2", "mejora_mae").toPandas().round(3).to_string(index=False))

## Conclusiones

Completar con los resultados. Resultados en validación (todo 2025, solo filas completas).

| Pregunta | Resultado |
|---|---|
| ¿Qué regla se equivoca menos? ¿Es la mediana de 4 semanas? | |
| Error de la regla a superar en el notebook 06 (MAE con su intervalo, WAPE, RMSE, sesgo) | |
| ¿El R² sirve para elegir entre reglas? | |
| ¿`perfil_historico` le gana a las reglas de semanas anteriores? ¿En qué tipo de día? | |
| ¿En qué tipo de día se equivoca más la mediana? ¿Hacia qué lado? | |
| ¿Los peores días son días especiales? ¿Cuáles? ¿Qué parte del error juntan? | |
| ¿Qué variables podrían evitar esos errores? | |

**Qué nos llevamos al modelo (notebook 06).** Completar: en qué días gana cada regla y cuánto bajaría el MAE si usáramos la mejor regla para cada tipo de día. Ahora train tiene un diciembre entero, así que en validación ya se puede ver cómo les va a las reglas en las fiestas de 2025.